# 01 Classical IDS (self-sufficient)

Upload **only** this notebook. Put the **8 CICIDS2017 flow CSVs** in **`MyDrive/cic_ids_data`** (Colab after mounting Drive) or in **`./cic_ids_data`** locally. Preprocessing + metrics live **inside this notebook** — there are **no** imports from the project’s `src` package (not `src/__init__.py`, not `src/ids_pipeline.py`, not `src/flow_autoencoder.py`). Artifacts: `/content/artifacts/` on Colab or **`./artifacts/`** next to the notebook locally.

**Data budget:** the next code cell sets `MAX_TOTAL_ROWS` for **~50k+ benign training rows** (75% of benign; 25% validation). **Local web UI** needs `artifacts/classical/oc_svm_model.pkl` (produced at the end of the run) plus the DL + hybrid files from notebooks 02 and 03 or the all-in-one notebook 03.

**Scapy / live demo:** see notebook 03 intro (run `uvicorn` with `sudo` if live sniff fails).

Optional GPU OC-SVM: `%pip install -q thundersvm` on a GPU runtime.


In [1]:
# --- Inline CIC helpers: standalone — do not import src.ids_pipeline, src.flow_autoencoder, or src.__init__ ---
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from tqdm.auto import tqdm
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
    from tqdm.auto import tqdm


CIC_DAY_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]


@dataclass
class PreprocessArtifacts:
    features: List[str]
    dropped_corr_cols: List[str]
    benign_train_indices: List[int]
    benign_val_indices: List[int]
    feature_impute_medians: List[float]


def _impute_features_to_float32(
    df: pd.DataFrame, features: List[str], medians: np.ndarray
) -> np.ndarray:
    x = (
        df.reindex(columns=features)
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy(dtype=np.float64)
    )
    med = np.asarray(medians, dtype=np.float64)
    med = np.where(np.isfinite(med), med, 0.0)
    nan_m = np.isnan(x)
    if nan_m.any():
        x = x.copy()
        x[nan_m] = np.broadcast_to(med, x.shape)[nan_m]
    np.nan_to_num(x, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return x.astype(np.float32, copy=False)


def norm_family(label: str) -> str:
    s = str(label).strip().replace("\x00", "")
    low = s.lower()
    if low == "benign":
        return "Benign"
    if "heartbleed" in low:
        return "Heartbleed"
    if "web attack" in low or "webattack" in low:
        return "WebAttack"
    if "portscan" in low:
        return "PortScan"
    if "dos" in low and "ddos" not in low:
        return "DoS"
    if "ddos" in low:
        return "DDoS"
    if "bot" in low:
        return "Bot"
    if "infiltration" in low:
        return "Infiltration"
    return s


def cic_report_row_name(raw_label: str) -> str:
    s = str(raw_label).strip()
    low = s.lower().replace("_", " ")
    if low == "benign":
        return "Benign (Validation)"
    if "ftp" in low and "patator" in low:
        return "FTP Bruteforce"
    if "ssh" in low and "patator" in low:
        return "SSH Bruteforce"
    if "slowloris" in low:
        return "DoS (Slowloris)"
    if "goldeneye" in low:
        return "DoS (GoldenEye)"
    if "hulk" in low:
        return "DoS (Hulk)"
    if "slowhttptest" in low or "slowhttps" in low:
        return "DoS (Slowhttps)"
    if "ddos" in low:
        return "DDoS"
    if "dos" in low and "ddos" not in low:
        return "DoS (other)"
    if "heartbleed" in low:
        return "Heartbleed"
    if "web" in low and ("bf" in low or "brute" in low):
        return "Web BF"
    if "xss" in low:
        return "Web XSS"
    if "sql" in low:
        return "Web SQL"
    if "infiltration" in low:
        return "Infiltration"
    if "bot" in low:
        return "Botnet"
    if "portscan" in low:
        return "PortScan"
    return s


def load_cic_data(
    data_dir: str | Path,
    chunk_rows: int = 200_000,
    max_total_rows: int | None = 900_000,
    max_benign_fraction: float = 0.7,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    data_dir = Path(data_dir)
    target_total = int(max_total_rows) if max_total_rows else None
    target_benign = None if target_total is None else int(target_total * max_benign_fraction)
    per_file_benign = (
        None if target_benign is None else int(np.ceil(target_benign / len(CIC_DAY_FILES)))
    )
    per_file_attack = (
        None
        if target_total is None
        else int(np.ceil((target_total - target_benign) / len(CIC_DAY_FILES)))
    )

    benign_parts: List[pd.DataFrame] = []
    attack_parts: List[pd.DataFrame] = []
    for file_name in tqdm(CIC_DAY_FILES, desc="CIC CSV files"):
        csv_path = data_dir / file_name
        kept_benign = 0
        kept_attack = 0
        for chunk in pd.read_csv(csv_path, low_memory=True, chunksize=chunk_rows):
            chunk.columns = [c.strip() for c in chunk.columns]
            raw = chunk["Label"].astype(str).str.strip()
            y = (raw.str.lower() != "benign").astype(np.int8)
            fam = raw.map(norm_family)
            x = chunk.drop(columns=["Label"], errors="ignore").select_dtypes(include=[np.number])
            if x.empty:
                continue
            x = x.replace([np.inf, -np.inf], np.nan).astype(np.float32, copy=False)
            x["y"] = y.values
            x["attack_family"] = fam.values
            x["cic_label"] = raw.values
            benign = x[x["y"] == 0]
            attack = x[x["y"] == 1]

            if per_file_benign is not None and len(benign):
                remaining = max(0, per_file_benign - kept_benign)
                if len(benign) > remaining:
                    benign = benign.iloc[rng.choice(len(benign), size=remaining, replace=False)]
            if per_file_attack is not None and len(attack):
                remaining = max(0, per_file_attack - kept_attack)
                if len(attack) > remaining:
                    attack = attack.iloc[rng.choice(len(attack), size=remaining, replace=False)]

            if len(benign):
                benign_parts.append(benign)
                kept_benign += len(benign)
            if len(attack):
                attack_parts.append(attack)
                kept_attack += len(attack)
            if per_file_benign is not None and per_file_attack is not None:
                if kept_benign >= per_file_benign and kept_attack >= per_file_attack:
                    break

    full = pd.concat([*benign_parts, *attack_parts], ignore_index=True)
    full = full.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return full


def drop_correlated_columns(df: pd.DataFrame, threshold: float = 0.9) -> tuple[pd.DataFrame, list[str]]:
    corr = df.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    dropped = [col for col in upper.columns if (upper[col] > threshold).any()]
    reduced = df.drop(columns=dropped)
    return reduced, dropped


def fit_preprocess_on_benign(
    df: pd.DataFrame,
    seed: int = 42,
    corr_threshold: float = 0.9,
    benign_train_size: float = 0.75,
) -> tuple[pd.DataFrame, pd.DataFrame, StandardScaler, PreprocessArtifacts]:
    feature_cols = [
        c
        for c in df.columns
        if c not in {"y", "attack_family", "cic_label", "report_name"}
    ]
    benign = df[df["y"] == 0].copy()
    benign = benign.dropna(axis=0)

    benign_train, benign_val = train_test_split(
        benign, train_size=benign_train_size, shuffle=True, random_state=seed
    )
    train_features = benign_train[feature_cols]
    train_features, dropped = drop_correlated_columns(train_features, threshold=corr_threshold)
    final_features = list(train_features.columns)
    median_vec = benign_train[final_features].median().to_numpy(dtype=np.float64)
    median_vec = np.where(np.isfinite(median_vec), median_vec, 0.0)

    scaler = StandardScaler()
    x_train = scaler.fit_transform(
        _impute_features_to_float32(benign_train[final_features], final_features, median_vec)
    )
    x_val = scaler.transform(
        _impute_features_to_float32(benign_val[final_features], final_features, median_vec)
    )

    artifacts = PreprocessArtifacts(
        features=final_features,
        dropped_corr_cols=dropped,
        benign_train_indices=benign_train.index.tolist(),
        benign_val_indices=benign_val.index.tolist(),
        feature_impute_medians=median_vec.tolist(),
    )
    return pd.DataFrame(x_train, columns=final_features), pd.DataFrame(
        x_val, columns=final_features
    ), scaler, artifacts


def iter_zero_day_folds(df: pd.DataFrame) -> Iterable[tuple[str, pd.DataFrame]]:
    attacks = df[df["y"] == 1].copy()
    col = "report_name" if "report_name" in df.columns else "attack_family"
    for family in sorted(attacks[col].unique()):
        yield str(family), attacks[attacks[col] == family].copy()


def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None = None) -> Dict[str, float]:
    y_true = np.asarray(y_true).ravel().astype(np.int8)
    y_pred = np.asarray(y_pred).ravel().astype(np.int8)
    negatives = np.sum(y_true == 0)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    fpr = float(fp / max(negatives, 1))
    specificity = float(tn / max(negatives, 1)) if negatives else 0.0
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_binary": float(f1_score(y_true, y_pred, zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "fallout_fpr": fpr,
        "specificity": specificity,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }
    if y_score is not None and len(np.unique(y_true)) > 1:
        out["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return out


def macro_f1_from_family_results(rows: Sequence[Dict[str, float]]) -> float:
    if not rows:
        return 0.0
    return float(np.mean([r["f1_macro"] for r in rows]))


def save_json(path: str | Path, payload: dict) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def save_run_config(path: str | Path, config: dict, preprocess: PreprocessArtifacts | None = None) -> None:
    payload = {"config": config}
    if preprocess is not None:
        payload["preprocess"] = asdict(preprocess)
    save_json(path, payload)


def robust_z(x: np.ndarray, ref: np.ndarray) -> np.ndarray:
    med = float(np.median(ref))
    mad = float(np.median(np.abs(ref - med))) + 1e-9
    return (np.asarray(x, dtype=np.float64) - med) / mad


def split_attack_families_for_calibration(
    df: pd.DataFrame,
    seed: int,
    calib_fraction: float = 0.5,
) -> Tuple[List[str], List[str]]:
    families = sorted(df.loc[df["y"] == 1, "attack_family"].unique().tolist())
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(families))
    n_calib = max(1, int(round(len(families) * calib_fraction)))
    calib_set = {families[i] for i in perm[:n_calib]}
    test_set = [f for f in families if f not in calib_set]
    calib_list = [f for f in families if f in calib_set]
    return calib_list, test_set

# --- Google Drive: dataset only ---
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    pass

import shutil

def resolve_cic_data_dir() -> Path:
    marker = Path("Monday-WorkingHours.pcap_ISCX.csv")
    drive_root = Path("/content/drive/MyDrive/cic_ids_data")
    local_cache = Path("/content/cic_ids_data_local_cache")

    def full_cic_set(root: Path) -> bool:
        return root.is_dir() and all((root / n).is_file() for n in CIC_DAY_FILES)

    # Colab: pandas read_csv over Drive FUSE often dies with errno 107 (transport endpoint). Copy once to VM disk.
    if Path("/content").is_dir() and full_cic_set(drive_root):
        local_cache.mkdir(parents=True, exist_ok=True)
        for name in tqdm(CIC_DAY_FILES, desc="Cache CIC CSVs to /content"):
            src, dst = drive_root / name, local_cache / name
            if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
                shutil.copy2(src, dst)
        print("Using VM-local CSV cache:", local_cache)
        return local_cache.resolve()

    if Path("/content").is_dir() and full_cic_set(local_cache):
        print("Using VM-local CSV cache:", local_cache)
        return local_cache.resolve()

    for base in (drive_root, Path.cwd() / "cic_ids_data", Path.cwd().parent / "cic_ids_data"):
        if base.is_dir() and (base / marker).is_file():
            return base.resolve()
    raise FileNotFoundError(
        "CICIDS CSVs not found. Colab: put the 8 flow CSVs in MyDrive/cic_ids_data. "
        "Local/Jupyter: put them in ./cic_ids_data next to this notebook."
    )


DATA_DIR = resolve_cic_data_dir()
ROOT_ARTIFACTS = Path("/content/artifacts") if Path("/content").is_dir() else (Path.cwd() / "artifacts")
ROOT_ARTIFACTS.mkdir(parents=True, exist_ok=True)
ART_DIR = ROOT_ARTIFACTS / "classical"
print("Resolved DATA_DIR:", DATA_DIR)
print("Artifacts root:", ROOT_ARTIFACTS.resolve())

import pickle
import torch
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

SEED = 42
TARGET_BENIGN_TRAIN_ROWS = 50_000
BENIGN_TRAIN_FRACTION = 0.75
MAX_BENIGN_FRACTION = 0.65
_MIN_TOTAL_FOR_TARGET_TRAIN = int(
    np.ceil(TARGET_BENIGN_TRAIN_ROWS / (BENIGN_TRAIN_FRACTION * MAX_BENIGN_FRACTION))
)
# Same ~50k benign-train target as notebooks 02–03
CHUNK_ROWS = 50_000
MAX_TOTAL_ROWS = max(110_000, _MIN_TOTAL_FOR_TARGET_TRAIN)
CORR_THRESHOLD = 0.9
SVM_NU_VALUES = [0.2, 0.15, 0.1]
RUN_IFOREST = True

print(
    f"Data budget: MAX_TOTAL_ROWS={MAX_TOTAL_ROWS:,} | expect benign_train ≲ "
    f"{int(MAX_TOTAL_ROWS * MAX_BENIGN_FRACTION * BENIGN_TRAIN_FRACTION):,} (target {TARGET_BENIGN_TRAIN_ROWS:,}+)"
)

_USE_THUNDER = False
_GPUOCSVM = None
if torch.cuda.is_available():
    try:
        from thundersvm import OneClassSvm as _GPUOCSVM
        _USE_THUNDER = _GPUOCSVM is not None
    except ImportError:
        _GPUOCSVM = None


def make_ocsvm(nu: float):
    if _USE_THUNDER and _GPUOCSVM is not None:
        return _GPUOCSVM(kernel='rbf', nu=nu)
    return OneClassSVM(
        kernel='rbf',
        nu=nu,
        cache_size=2048,
        tol=1e-2,
        verbose=True,
    )

OCSVM_FIT_MAX_SAMPLES = None if _USE_THUNDER else 10_000

print('DATA_DIR=', DATA_DIR)
print('OC-SVM backend:', 'ThunderSVM (GPU)' if _USE_THUNDER else 'sklearn (CPU). Optional: pip install thundersvm on GPU runtime')
if OCSVM_FIT_MAX_SAMPLES:
    print(
        'OC-SVM sklearn: fitting on at most',
        OCSVM_FIT_MAX_SAMPLES,
        'benign rows (200k+ RBF often looks frozen). Raise cap only with ThunderSVM GPU.',
    )
if torch.cuda.is_available():
    print('CUDA:', torch.cuda.get_device_name(0))


Mounted at /content/drive


Cache CIC CSVs to /content:   0%|          | 0/8 [00:00<?, ?it/s]

Using VM-local CSV cache: /content/cic_ids_data_local_cache
Resolved DATA_DIR: /content/cic_ids_data_local_cache
Artifacts root: /content/artifacts
DATA_DIR= /content/cic_ids_data_local_cache
OC-SVM backend: sklearn (CPU). Optional: pip install thundersvm on GPU runtime
OC-SVM sklearn: fitting on at most 10000 benign rows (200k+ RBF often looks frozen). Raise cap only with ThunderSVM GPU.
CUDA: Tesla T4


In [2]:
df = load_cic_data(
    data_dir=DATA_DIR,
    chunk_rows=CHUNK_ROWS,
    max_total_rows=MAX_TOTAL_ROWS,
    max_benign_fraction=MAX_BENIGN_FRACTION,
    seed=SEED,
)
print('Rows loaded:', len(df), 'benign~', int((df["y"] == 0).sum()))
df['report_name'] = df['cic_label'].astype(str).map(cic_report_row_name)

x_benign_train, x_benign_val, scaler, prep = fit_preprocess_on_benign(
    df,
    seed=SEED,
    corr_threshold=CORR_THRESHOLD,
    benign_train_size=BENIGN_TRAIN_FRACTION,
)
print('Benign train / val rows (features):', len(x_benign_train), len(x_benign_val))
features = prep.features

med = np.asarray(prep.feature_impute_medians, dtype=np.float64)
x_val_benign = scaler.transform(
    _impute_features_to_float32(df.loc[prep.benign_val_indices, features], features, med)
)
y_val_benign = np.zeros(len(x_val_benign), dtype=np.int8)

_xb_full = x_benign_train.to_numpy(np.float32)
_fit_rng = np.random.default_rng(SEED)
if (not _USE_THUNDER) and OCSVM_FIT_MAX_SAMPLES is not None and len(_xb_full) > OCSVM_FIT_MAX_SAMPLES:
    _idx = _fit_rng.choice(len(_xb_full), size=OCSVM_FIT_MAX_SAMPLES, replace=False)
    _xb_ocsvm_fit = _xb_full[_idx]
    print('OC-SVM fit benign rows:', len(_xb_ocsvm_fit), '/', len(_xb_full))
else:
    _xb_ocsvm_fit = _xb_full

svm_rows = []
best = {'nu': None, 'macro_f1': -1.0, 'results': None, 'model': None}
for nu in tqdm(SVM_NU_VALUES, desc='OC-SVM ν sweep'):
    svm = make_ocsvm(nu)
    svm.fit(_xb_ocsvm_fit)
    family_results = []
    for family, atk_df in tqdm(list(iter_zero_day_folds(df)), desc=f'families ν={nu}', leave=False):
        x_attack = scaler.transform(_impute_features_to_float32(atk_df[features], features, med))
        x_eval = np.vstack([x_val_benign, x_attack])
        y_true = np.concatenate([y_val_benign, np.ones(len(x_attack), dtype=np.int8)])
        score = -svm.decision_function(x_eval)
        y_pred = (score > 0).astype(np.int8)
        metrics = binary_metrics(y_true, y_pred, score)
        metrics['family'] = family
        family_results.append(metrics)
    sb = -svm.decision_function(x_val_benign)
    benign_val_acc = float(np.mean((sb > 0).astype(np.int8) == 0))
    macro_f1 = macro_f1_from_family_results(family_results)
    svm_rows.append(
        {
            'nu': nu,
            'macro_f1': macro_f1,
            'per_family': family_results,
            'benign_val_accuracy': benign_val_acc,
        }
    )
    if macro_f1 > best['macro_f1']:
        best = {'nu': nu, 'macro_f1': macro_f1, 'results': family_results, 'model': svm}

iforest_payload = None
if RUN_IFOREST:
    iso = IsolationForest(
        random_state=SEED, n_estimators=300, contamination=0.1, n_jobs=-1
    )
    iso.fit(x_benign_train.to_numpy(np.float32))
    iso_rows = []
    for family, atk_df in tqdm(list(iter_zero_day_folds(df)), desc='IF families', leave=False):
        x_attack = scaler.transform(_impute_features_to_float32(atk_df[features], features, med))
        x_eval = np.vstack([x_val_benign, x_attack])
        y_true = np.concatenate([y_val_benign, np.ones(len(x_attack), dtype=np.int8)])
        score = -iso.score_samples(x_eval)
        thr = np.quantile(score[:len(y_val_benign)], 0.95)
        y_pred = (score >= thr).astype(np.int8)
        metrics = binary_metrics(y_true, y_pred, score)
        metrics['family'] = family
        iso_rows.append(metrics)
    iforest_payload = {
        'macro_f1': macro_f1_from_family_results(iso_rows),
        'per_family': iso_rows,
    }

ART_DIR.mkdir(parents=True, exist_ok=True)
with (ART_DIR / 'oc_svm_model.pkl').open('wb') as f:
    pickle.dump(
        {
            'model': best['model'],
            'features': features,
            'scaler': scaler,
            'best_nu': best['nu'],
            'svm_search': svm_rows,
        },
        f,
    )

save_json(
    ART_DIR / 'classical_results.json',
    {
        'best_nu': best['nu'],
        'best_macro_f1': best['macro_f1'],
        'one_class_svm_per_family': best['results'],
        'one_class_svm_search': svm_rows,
        'isolation_forest': iforest_payload,
    },
)
save_run_config(
    ART_DIR / 'run_config.json',
    {
        'seed': SEED,
        'data_dir': str(DATA_DIR),
        'chunk_rows': CHUNK_ROWS,
        'max_total_rows': MAX_TOTAL_ROWS,
        'max_benign_fraction': MAX_BENIGN_FRACTION,
        'corr_threshold': CORR_THRESHOLD,
        'svm_nu_values': SVM_NU_VALUES,
        'run_iforest': RUN_IFOREST,
    },
    prep,
)

print('Best One-Class SVM nu:', best['nu'])
print('Best macro F1:', round(best['macro_f1'], 4))
if iforest_payload:
    print('IsolationForest macro F1:', round(iforest_payload['macro_f1'], 4))

print(
    '\n[Web UI] Copy into repo: artifacts/classical/oc_svm_model.pkl + '
    'classical_results.json (optional). You still need notebook 02/03 outputs for full hybrid.\n'
)


CIC CSV files:   0%|          | 0/8 [00:00<?, ?it/s]

Rows loaded: 12818 benign~ 9104


OC-SVM ν sweep:   0%|          | 0/3 [00:00<?, ?it/s]

[LibSVM]

families ν=0.2:   0%|          | 0/7 [00:00<?, ?it/s]

...

families ν=0.15:   0%|          | 0/7 [00:00<?, ?it/s]

[LibSVM]

families ν=0.1:   0%|          | 0/7 [00:00<?, ?it/s]

IF families:   0%|          | 0/7 [00:00<?, ?it/s]

Best One-Class SVM nu: 0.1
Best macro F1: 0.562
IsolationForest macro F1: 0.5294


### Table 2 (CICIDS2017 One-Class SVM)
Per-class **accuracy** for each ν (Benign row = validation benign correctly classified as inlier). Primary tuning metric in other cells: **macro-F1** (mean of per-family `f1_macro`).


In [3]:
# --- Table 2: CICIDS2017 One-Class SVM — accuracy per class × ν (matches coursework layout) ---
_figd = ART_DIR / "figures"
_figd.mkdir(parents=True, exist_ok=True)
_classes_t2 = ["Benign (Validation)"] + sorted(
    {m["family"] for row in svm_rows for m in row["per_family"]}
)
_rows_t2 = []
for cls in _classes_t2:
    rd = {"Class": cls}
    for row in svm_rows:
        nu = row["nu"]
        if cls == "Benign (Validation)":
            rd[f"ν={nu}"] = round(100.0 * row["benign_val_accuracy"], 2)
        else:
            fam = next((m for m in row["per_family"] if m["family"] == cls), None)
            rd[f"ν={nu}"] = round(100.0 * fam["accuracy"], 2) if fam else np.nan
    _rows_t2.append(rd)
table2_svm = pd.DataFrame(_rows_t2)
table2_svm.to_csv(_figd / "table2_svm_accuracy_cicids.csv", index=False)
display(table2_svm)


,Class,ν=0.2,ν=0.15,ν=0.1
0,Benign (Validation),81.45,86.90,91.12
1,Botnet,66.69,70.39,73.55
2,DDoS,79.16,83.45,86.77
3,DoS (Slowloris),76.59,80.51,83.48
4,FTP Bruteforce,73.41,68.59,71.88
5,Infiltration,81.57,86.89,91.04
6,PortScan,64.79,68.98,72.23
7,Web BF,65.30,69.60,72.89


In [4]:
pd.DataFrame(best['results']).sort_values('recall', ascending=False).head(10)

,accuracy,precision,recall,f1_binary,f1_macro,fallout_fpr,specificity,tp,fp,tn,fn,roc_auc,family
4,0.910428,0.133047,0.861111,0.230483,0.591465,0.088791,0.911209,31,202,2073,5,0.924005,Infiltration
1,0.867729,0.681890,0.706362,0.693910,0.804773,0.088791,0.911209,433,202,2073,180,0.783609,DDoS
2,0.834834,0.625926,0.551387,0.586297,0.741558,0.088791,0.911209,338,202,2073,275,0.899979,DoS (Slowloris)
0,0.735457,0.201581,0.083197,0.117783,0.481091,0.088791,0.911209,51,202,2073,562,0.647113,Botnet
6,0.728878,0.136752,0.052202,0.075561,0.458353,0.088791,0.911209,32,202,2073,581,0.606428,Web BF
5,0.722299,0.060465,0.021207,0.031401,0.434658,0.088791,0.911209,13,202,2073,600,0.598524,PortScan
3,0.718837,0.014634,0.004894,0.007335,0.421780,0.088791,0.911209,3,202,2073,610,0.691986,FTP Bruteforce
